In [40]:
import numpy as np

from qiskit import QuantumCircuit, transpile, generate_preset_pass_manager
from qiskit.circuit.library import CXGate,  PauliEvolutionGate, QAOAAnsatz
from qiskit.quantum_info import SparsePauliOp

from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import InverseCancellation, CommutativeCancellation
from qopt_best_practices.transpilation.swap_cancellation_pass import SwapToFinalMapping

from qiskit_aer import AerSimulator
from qiskit_aer.backends.backendconfiguration import AerBackendConfiguration

from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy, FindCommutingPauliEvolutionsMulti
from qiskit_qaoa.utils.commuting_gate_router_rzz import CommutingGateRouterRzz

In [41]:
def print_circuit_info(qc: QuantumCircuit, circuit_name):
    print(f'{circuit_name} has {qc.num_qubits} qubits')
    print(f'{circuit_name} has {qc.num_nonlocal_gates()} non-local gates and {qc.depth(lambda instr: len(instr.qubits) > 1)} non-local depth')
    print(f'{circuit_name} contains {list(qc.count_ops().keys())} gates.')

In [42]:
num_qubits = 5
ess = ExtendedSwapStrategy.from_line(range(num_qubits), 3)

operators = ['ZZZZZ','ZZZ', 'ZZZ', 'ZZZZ']
qubits = [[0,1,2,3,4],[0,1,2],[1,2,4], [1,2,3,4]]
=> (22, 21, 12) depth,  (24, 22, 14) count

operators = ['ZZZZZ','ZZZZ','ZZZ', 'ZZZ', 'ZZZZ']
qubits = [[0,1,2,3,4], [0,1,2,3],[1,2,4],[0,1,2], [1,2,3,4]]
=> (31,28,16) depth, (32,30,19) count

In [43]:
qubits = [[0,1,2,3,4],[0,1,2],[1,2,4], [1,2,3,4]]
operators = ['Z'*len(op) for op in qubits]
coeffs = [1] * len(operators)

hamiltonian = SparsePauliOp.from_sparse_list(
    zip(operators, qubits, coeffs),
    num_qubits
)

In [44]:
basis_gates=["sx", "x", "rz", "rzz", "cz", "id", "cx", "swap"]
method = 'unitary'
backend_options = dict(
    method=method,
    device='CPU',
    precision='single',
    basis_gates=basis_gates
)

config = AerSimulator._DEFAULT_CONFIGURATION
config["n_qubits"] = num_qubits
config = AerBackendConfiguration.from_dict(config)
backend = AerSimulator(configuration=config, coupling_map=ess._coupling_map, **backend_options)
backend.set_option("n_qubits", num_qubits)
print(f'Num qubits in backend: {backend.configuration().to_dict()["n_qubits"]}')

Num qubits in backend: 5


In [45]:
default = QAOAAnsatz(hamiltonian, reps=1, initial_state=QuantumCircuit(num_qubits), mixer_operator=QuantumCircuit(num_qubits))
t_default = transpile(default, backend, optimization_level=0,basis_gates=basis_gates)
print_circuit_info(t_default, 'Default no opt')
t_default.draw(fold=-1)

Default no opt has 5 qubits
Default no opt has 24 non-local gates and 22 non-local depth
Default no opt contains ['cx', 'rz', 'swap'] gates.


/nfs/users/nfs_j/jc59/quantumwork/pangenome/.venv/lib/python3.10/site-packages/qiskit/compiler/transpiler.py:269: UserWarning: Providing `coupling_map` and/or `basis_gates` along with `backend` is not recommended, as this will invalidate the backend's gate durations and error rates.
  pm = generate_preset_pass_manager(


┌───┐┌────────────┐┌───┐               ┌───┐┌────────────┐┌───┐                                                                                      
q_0 -> 0 ───────────────┤ X ├┤ Rz(2*γ[0]) ├┤ X ├───────────────┤ X ├┤ Rz(2*γ[0]) ├┤ X ├──────────────────────────────────────────────────────────────────────────────────────
                   ┌───┐└─┬─┘└────────────┘└─┬─┘┌───┐     ┌───┐└─┬─┘└────────────┘└─┬─┘┌───┐     ┌───┐┌────────────┐┌───┐                  ┌───┐┌────────────┐┌───┐          
q_1 -> 1 ──────────┤ X ├──■──────────────────■──┤ X ├─────┤ X ├──■──────────────────■──┤ X ├─────┤ X ├┤ Rz(2*γ[0]) ├┤ X ├──────────────────┤ X ├┤ Rz(2*γ[0]) ├┤ X ├──────────
              ┌───┐└─┬─┘                        └─┬─┘┌───┐└─┬─┘                        └─┬─┘┌───┐└─┬─┘└────────────┘└─┬─┘┌───┐        ┌───┐└─┬─┘└────────────┘└─┬─┘┌───┐     
q_2 -> 2 ─────┤ X ├──■────────────────────────────■──┤ X ├──■────────────────────────────■──┤ X ├──■──────────────────■──┤ X ├────────┤ X ├──■──────────────────■──┤ X ├─────
         ┌───┐└─┬─┘                                  └─┬─┘┌───┐                             └─┬─┘                        └─┬─┘        └─┬─┘                        └─┬─┘┌───┐
q_3 -> 3 ┤ X ├──■──────────────────────────────────────■──┤ X ├──X────────────────────────────■────────────────────────────■────■───X───■────────────────────────────■──┤ X ├
         └─┬─┘                                            └─┬─┘  │                                                            ┌─┴─┐ │                                   └─┬─┘
q_4 -> 4 ──■────────────────────────────────────────────────■────X────────────────────────────────────────────────────────────┤ X ├─X─────────────────────────────────────■──
                                                                                                                              └───┘

In [46]:
default = QAOAAnsatz(hamiltonian, reps=1, initial_state=QuantumCircuit(num_qubits), mixer_operator=QuantumCircuit(num_qubits))
t_default = transpile(default, backend, optimization_level=3,basis_gates=basis_gates)
print_circuit_info(t_default, 'Default w/ layout')
t_default.draw(fold=-1)

Default w/ layout has 5 qubits
Default w/ layout has 22 non-local gates and 21 non-local depth
Default w/ layout contains ['cx', 'rz', 'swap'] gates.


q_4 -> 0 ──■────────────────────────────────────────────────X──────────────────────────────────────────────────────────────────X─────────────────────────────────────■──
         ┌─┴─┐                                              │                                                                  │                                   ┌─┴─┐
q_3 -> 1 ┤ X ├──■──────────────────────────────────────■────X─────────────────────────────────■────────────────────────────■───X───■────────────────────────────■──┤ X ├
         └───┘┌─┴─┐                                  ┌─┴─┐                                  ┌─┴─┐                        ┌─┴─┐   ┌─┴─┐                        ┌─┴─┐└───┘
q_2 -> 2 ─────┤ X ├──■────────────────────────────■──┤ X ├──■────────────────────────────■──┤ X ├──■──────────────────■──┤ X ├───┤ X ├──■──────────────────■──┤ X ├─────
              └───┘┌─┴─┐                        ┌─┴─┐└───┘┌─┴─┐                        ┌─┴─┐└───┘┌─┴─┐┌────────────┐┌─┴─┐└───┘   └───┘┌─┴─┐┌────────────┐┌─┴─┐└───┘     
q_1 -> 3 ──────────┤ X ├──■──────────────────■──┤ X ├─────┤ X ├──■──────────────────■──┤ X ├─────┤ X ├┤ Rz(2*γ[0]) ├┤ X ├─────────────┤ X ├┤ Rz(2*γ[0]) ├┤ X ├──────────
                   └───┘┌─┴─┐┌────────────┐┌─┴─┐└───┘     └───┘┌─┴─┐┌────────────┐┌─┴─┐└───┘     └───┘└────────────┘└───┘             └───┘└────────────┘└───┘          
q_0 -> 4 ───────────────┤ X ├┤ Rz(2*γ[0]) ├┤ X ├───────────────┤ X ├┤ Rz(2*γ[0]) ├┤ X ├─────────────────────────────────────────────────────────────────────────────────
                        └───┘└────────────┘└───┘               └───┘└────────────┘└───┘

In [47]:
default = QAOAAnsatz(hamiltonian, reps=1, initial_state=QuantumCircuit(num_qubits), mixer_operator=QuantumCircuit(num_qubits))
generic_pm = generate_preset_pass_manager(optimization_level=3, backend=backend, basis_gates=basis_gates)
generic_pm.layout = None
t_default = generic_pm.run(default)
print_circuit_info(t_default, 'Default w/o layout')
t_default.draw(fold=-1)

Default w/o layout has 5 qubits
Default w/o layout has 22 non-local gates and 21 non-local depth
Default w/o layout contains ['cx', 'rz', 'swap'] gates.


/tmp/ipykernel_2535594/4184589419.py:2: UserWarning: Providing `coupling_map` and/or `basis_gates` along with `backend` is not recommended, as this will invalidate the backend's gate durations and error rates.
  generic_pm = generate_preset_pass_manager(optimization_level=3, backend=backend, basis_gates=basis_gates)


┌───┐┌────────────┐┌───┐               ┌───┐┌────────────┐┌───┐                                                                                 
q_0: ───────────────┤ X ├┤ Rz(2*γ[0]) ├┤ X ├───────────────┤ X ├┤ Rz(2*γ[0]) ├┤ X ├─────────────────────────────────────────────────────────────────────────────────
               ┌───┐└─┬─┘└────────────┘└─┬─┘┌───┐     ┌───┐└─┬─┘└────────────┘└─┬─┘┌───┐     ┌───┐┌────────────┐┌───┐             ┌───┐┌────────────┐┌───┐          
q_1: ──────────┤ X ├──■──────────────────■──┤ X ├─────┤ X ├──■──────────────────■──┤ X ├─────┤ X ├┤ Rz(2*γ[0]) ├┤ X ├─────────────┤ X ├┤ Rz(2*γ[0]) ├┤ X ├──────────
          ┌───┐└─┬─┘                        └─┬─┘┌───┐└─┬─┘                        └─┬─┘┌───┐└─┬─┘└────────────┘└─┬─┘┌───┐   ┌───┐└─┬─┘└────────────┘└─┬─┘┌───┐     
q_2: ─────┤ X ├──■────────────────────────────■──┤ X ├──■────────────────────────────■──┤ X ├──■──────────────────■──┤ X ├───┤ X ├──■──────────────────■──┤ X ├─────
     ┌───┐└─┬─┘                                  └─┬─┘                                  └─┬─┘                        └─┬─┘   └─┬─┘                        └─┬─┘┌───┐
q_3: ┤ X ├──■──────────────────────────────────────■────X─────────────────────────────────■────────────────────────────■───X───■────────────────────────────■──┤ X ├
     └─┬─┘                                              │                                                                  │                                   └─┬─┘
q_4: ──■────────────────────────────────────────────────X──────────────────────────────────────────────────────────────────X─────────────────────────────────────■──

In [48]:
pm = PassManager(
    [
        FindCommutingPauliEvolutionsMulti(), 
        CommutingGateRouterRzz(
            ess,
            max_layers=0,
            perform_extra_swaps=True
        ),
        SwapToFinalMapping(),
        InverseCancellation(gates_to_cancel=[CXGate()]),
        CommutativeCancellation(basis_gates=["cx", "swap", "rz", "rzz"]),
        InverseCancellation(gates_to_cancel=[CXGate()]),
    ]
)

In [49]:
qc = QuantumCircuit(num_qubits)
qc.append(PauliEvolutionGate(hamiltonian), range(num_qubits))
tqc = pm.run(qc)
print_circuit_info(tqc, 'Rzz circuit')
tqc.draw(fold=-1)

Max layers needed to apply swap decompose: 0
(0, 1, 2) (0, 1, 2, 3, 4)
{3, 4} {0: {0}, 1: {1}, 2: {2}, 3: {3}, 4: {4}} False False True
{3, 4} {0: {0}, 1: {1}, 2: {2}, 3: {3}, 4: {4}} True True True
Testing subsets for (0, 1, 2, 3, 4): [(1, 2, 3, 4)]
(0, 1, 2, 3, 4) (1, 2, 3, 4)
{0} {0: {0}, 1: {0, 1}, 2: {2, 3, 4}, 3: {3, 4}, 4: {4}} True True True
Starting end of layer seek for impossible gates
Testing subsets for (1, 2, 3, 4): [(1, 2, 4)]
(1, 2, 3, 4) (1, 2, 4)
{3} {0: {0}, 1: {1}, 2: {2, 3, 4}, 3: {3, 4}, 4: {4}} True True True
Got interaction from impossible gates
Gates we cannot directly implement: 0
[]
Transpiling accumulator


Rzz circuit has 5 qubits
Rzz circuit has 14 non-local gates and 12 non-local depth
Rzz circuit contains ['cx', 'rzz'] gates.


q_0 -> 0 ──■─────────────────────────■───────────────────────────────────────────
         ┌─┴─┐                     ┌─┴─┐                                         
q_1 -> 1 ┤ X ├─■────────────■──────┤ X ├─■────────────■──────────────────────────
         └───┘ │ZZ(2) ┌───┐ │ZZ(2) └───┘ │ZZ(2) ┌───┐ │ZZ(2) ┌───┐     ┌───┐     
q_2 -> 2 ──────■──────┤ X ├─■────────────■──────┤ X ├─■──────┤ X ├─────┤ X ├─────
         ┌───┐        └─┬─┘ ┌───┐               └─┬─┘        └─┬─┘┌───┐└─┬─┘┌───┐
q_3 -> 3 ┤ X ├──────────■───┤ X ├─────────────────■────────────■──┤ X ├──■──┤ X ├
         └─┬─┘              └─┬─┘                                 └─┬─┘     └─┬─┘
q_4 -> 4 ──■──────────────────■─────────────────────────────────────■─────────■──